# Tabela de harmonização LULC 

Harmonização das classes de uso e ocupação do solo (COS) para os anos de 1995, 2007, 2010, 2015 e 2018.

O objectivo é construir uma legenda harmonizada comum entre os vários anos, de forma a permitir a comparação temporal e a  utilização destas classes no modelo de suscetibilidade. 

* A harmonização é feita a partir do nível 4 da COS, após exclusão prévia das classes artificiais e das classes de água com base no nível 1.

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [ ]:
fld = "/code/data/raw/lulc"
out = "/code/data/processed/pnse/lulc/harmonized_tables"
Path(out).mkdir(parents=True, exist_ok=True) 

cos = {
    1995: {
        "file": f"{fld}/COS1995v2-S1.gpkg",
        "layer": "COS1995v2",
        "n1": "COS95n1_C",
        "code": "COS95n4_C",
        "label": "COS95n4_L",
    },
    2007: {
        "file": f"{fld}/COS2007v3-S1.gpkg",
        "layer": "COS2007v3",
        "n1": "COS07n1_C",
        "code": "COS07n4_C",
        "label": "COS07n4_L",
    },
    2010: {
        "file": f"{fld}/COS2010v2-S1.gpkg",
        "layer": "COS2010v2",
        "n1": "COS10n1_C",
        "code": "COS10n4_C",
        "label": "COS10n4_L",
    },
    2015: {
        "file": f"{fld}/COS2015v2-S1.gpkg",
        "layer": "COS2015v2",
        "n1": "COS15n1_C",
        "code": "COS15n4_C",
        "label": "COS15n4_L",
    },
    2018: {
        "file": f"{fld}/COS2018v2-S1.gpkg",
        "layer": "COS2018v2",
        "n1": "COS18n1_C",
        "code": "COS18n4_C",
        "label": "COS18n4_L",
    },
}

In [ ]:
#inspecionar campos
for year, cfg in cos.items():
    gdf = gpd.read_file(cfg["file"], layer=cfg["layer"], rows=5)
    print(year, gdf.columns.tolist())

## Exclusão de áreas artificiais e água

Nesta fase são excluídas todas as classes cujo nível 1 (`n1`) corresponde a:

- `1` — territórios artificializados
- `9` — massas de água

In [ ]:
classes_n4 = {}

for year in cos:
    file = cos[year]["file"]
    layer = cos[year]["layer"]
    n1 = cos[year]["n1"]
    code = cos[year]["code"]
    label = cos[year]["label"]

    gdf = gpd.read_file(file, layer=layer)[[n1, code, label]].copy()

    gdf = gdf[~gdf[n1].astype(str).isin(["1", "9"])] #níveis da cos estão em string (verificado nas Especificações Técnicas da COS)

    df = (
        gdf[[code, label]]
        .drop_duplicates()
        .sort_values([code, label])
        .reset_index(drop=True)
    )

    df.columns = ["code", "label"]
    df["year"] = year
    df = df[["year", "code", "label"]]

    classes_n4[year] = df

    print(year, len(df))
    display(df.head())

## Comparação das classes de nível 4 entre anos

In [ ]:
dfs = []

for year in classes_n4:
    df = classes_n4[year][["code", "label"]].copy()
    df.columns = [f"code_{year}", "label"]
    dfs.append(df)

df_cmp = dfs[0].copy()

for df in dfs[1:]:
    df_cmp = df_cmp.merge(df, on="label", how="outer")

df_cmp = df_cmp.sort_values("label").reset_index(drop=True)

df_cmp.head(90)

In [ ]:
#Ver só as classes que aparecem em todos os anos
cols = [f"code_{year}" for year in classes_n4]

df_cmp_all = df_cmp.dropna(subset=cols).copy()
df_cmp_all = df_cmp_all.reset_index(drop=True)

df_cmp_all.head(100)

In [ ]:
## ver classes que faltam em pelo menos um ano
df_cmp_diff = df_cmp[df_cmp[cols].isna().any(axis=1)].copy()
df_cmp_diff = df_cmp_diff.reset_index(drop=True)

df_cmp_diff.head(100)

In [ ]:
print("Total de classes na comparação:", len(df_cmp))
print("Classes presentes em todos os anos:", len(df_cmp_all))
print("Classes com diferenças entre anos:", len(df_cmp_diff))

## Construção da tabela de harmonização

In [ ]:
# juntar todas as classes
df_harm = pd.concat(classes_n4.values(), ignore_index=True)
df_harm = df_harm.sort_values(["year", "code", "label"]).reset_index(drop=True)

df_harm.columns = ["year", "code_original", "label_original"]

df_harm.head(50)

In [ ]:
# criar novas colunas
df_harm["label_harmonizada"] = df_harm["label_original"]
df_harm["incluir_modelo"] = 1
df_harm["obs"] = "Mantido directamente"

## Exclusão de classes que não entram no modelo

In [ ]:
# excluir classes do nivel 4 que não se enquadram no contexto
labels_excluir = [
    "Pauis",
    "Sapais",
    "Zonas entremarés",
    "Praias, dunas e areais",
    "Praias, dunas e areais costeiros",
    "Praias, dunas e areais interiores",
]

df_harm.loc[df_harm["label_original"].isin(labels_excluir), "incluir_modelo"] = 0
df_harm.loc[df_harm["label_original"].isin(labels_excluir), "obs"] = "Excluir do modelo"

## Agregação das classes 

In [ ]:
# agregar pastagens
labels_pastagens = [
    "Pastagens",
    "Pastagens espontâneas",
    "Pastagens melhoradas",
]

df_harm.loc[df_harm["label_original"].isin(labels_pastagens), "label_harmonizada"] = "Pastagens"
df_harm.loc[df_harm["label_original"].isin(labels_pastagens), "obs"] = "Agregado na classe Pastagens"

In [ ]:
#filtrar modelo
df_harm = df_harm[df_harm["incluir_modelo"] == 1].copy()

df_leg = (
    df_harm[["label_harmonizada"]]
    .drop_duplicates()
    .sort_values("label_harmonizada")
    .reset_index(drop=True)
)

df_leg["id_harm"] = range(1, len(df_leg) + 1)


# juntar id_harm à tabela completa
df_harm = df_harm.merge(
    df_leg[["label_harmonizada", "id_harm"]],
    on="label_harmonizada",
    how="left"
)

df_harm["id_harm"] = df_harm["id_harm"].astype("Int64")

## Exportação da tabela de harmonização

In [ ]:
#guardar tabela de harmonização
df_harm.to_csv(f"{out}/tabela_harmonizacao.csv", index=False)

In [ ]:
df_harm.head()